In [1]:
!pip install -q pandas openpyxl chromadb sentence-transformers boto3 gradio ipywidgets



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import pandas as pd
import re, json, uuid, csv, os
from datetime import datetime
from collections import Counter
import boto3
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

# ── CONFIG ──────────────────────────────────────────
FILEPATH          = 'Updated_DataLake_Questionnaire.xlsx'
CHROMA_DIR        = './mbp_rag_db_advanced'
COLLECTION_NAME   = 'mbp_rag_collection_v2'
EMBED_MODEL       = 'all-MiniLM-L6-v2'
RERANK_MODEL      = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
BEDROCK_REGION    = 'ap-south-1'
BEDROCK_MODEL_ID  = 'anthropic.claude-3-haiku-20240307-v1:0'
TOP_K_RETRIEVE    = 5
TOP_K_RERANK      = 3
LOG_FILE          = 'chat_log.csv'

print('Config loaded.')


Config loaded.


In [3]:
def normalize_key(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'[^a-z0-9_]', '_', text)
    return re.sub(r'_+', '_', text).strip('_')

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    def clean(col):
        col = str(col).strip().lower().replace(' ', '_')
        col = re.sub(r'[^a-z0-9_]', '_', col)
        return re.sub(r'_+', '_', col).strip('_')
    df.columns = [clean(c) for c in df.columns]
    return df

def build_standardized_file_key(filename: str):
    if pd.isna(filename): return None
    fn = str(filename).lower().strip()
    fn = re.sub(r'\.txt|\.csv', '', fn)
    fn = fn.replace('-', '_')
    fn = re.sub(r'\d{8}', '', fn)
    fn = re.sub(r'yyyy_mm_dd|yyyymmdd', '', fn)
    return re.sub(r'_+', '_', fn).strip('_')

print('Helpers ready.')


Helpers ready.


In [5]:
# ── VERIFY SHEET NAMES BEFORE LOADING ──
import pandas as pd
xl = pd.ExcelFile(FILEPATH)
print('Sheets found in your Excel file:')
for s in xl.sheet_names:
    print(f'  → {s}')

# Expected sheet names used in Cell 4:
SHEET_QUESTIONNAIRE  = 'Questionnaire'
SHEET_FILE_INFO      = 'File Information if File Based'
SHEET_MAPPING        = 'Metadata_OR_Mapping'

# Validate
missing = [s for s in [SHEET_QUESTIONNAIRE, SHEET_FILE_INFO, SHEET_MAPPING]
           if s not in xl.sheet_names]
if missing:
    print(f'\n❌ MISSING SHEETS: {missing}')
    print('Update SHEET_QUESTIONNAIRE / SHEET_FILE_INFO / SHEET_MAPPING above to match your actual sheet names.')
else:
    print('\n✅ All 3 sheets found. Safe to proceed.')


Sheets found in your Excel file:
  → Questionnaire
  → File Information if File Based
  → Metadata_OR_Mapping

✅ All 3 sheets found. Safe to proceed.


In [7]:
def load_questionnaire_sheet(filepath):
    df = pd.read_excel(filepath, sheet_name='Questionnaire', header=None)
    df = df.dropna(how='all').reset_index(drop=True)
    dflower = df.apply(lambda col: col.astype(str).str.strip().str.lower())
    header_mask = dflower.apply(lambda r: {'field','description','value'}.issubset(set(r)), axis=1)
    if not header_mask.any(): raise ValueError('Header row not found in Questionnaire')
    header_idx = header_mask.idxmax()
    df.columns = df.iloc[header_idx].astype(str).str.strip().str.lower()
    df = df.iloc[header_idx+1:].reset_index(drop=True)
    category_rows = df[df['field'].astype(str).str.strip().str.lower() == 'category']
    split_idx = category_rows.index[0]
    project_df = df.iloc[:split_idx].copy().reset_index(drop=True)
    questionnaire_df = df.iloc[split_idx+1:].copy().reset_index(drop=True)
    questionnaire_df.columns = ['category', 'description', 'response']
    questionnaire_df = questionnaire_df.apply(
        lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))
    return project_df, questionnaire_df


def load_file_information_sheet(filepath):
    raw = pd.read_excel(filepath, sheet_name='File Information if File Based', header=None)
    raw = raw.dropna(how='all').reset_index(drop=True)
    dflower = raw.apply(lambda col: col.astype(str).str.strip().str.lower())
    header_mask = dflower.apply(lambda r: 'file name' in list(r), axis=1)
    if not header_mask.any(): raise ValueError('Header not found in File Information')
    header_idx = header_mask.idxmax()
    df = raw.iloc[header_idx:].reset_index(drop=True)
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    df = normalize_columns(df)
    df['standardized_file_key'] = df['file_name'].apply(build_standardized_file_key)
    return df.reset_index(drop=True)


def load_metadata_mapping_sheet(filepath, file_information_df):
    raw = pd.read_excel(filepath, sheet_name='Metadata_OR_Mapping', header=None)
    raw = raw.dropna(how='all').reset_index(drop=True)
    dflower = raw.apply(lambda col: col.astype(str).str.strip().str.lower())
    header_mask = dflower.apply(lambda r: 'actual file name' in list(r), axis=1)
    if not header_mask.any(): raise ValueError('Header not found in Metadata_OR_Mapping')
    header_idx = header_mask.idxmax()
    df = raw.iloc[header_idx:].reset_index(drop=True)
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    df = normalize_columns(df)
    df['standardized_file_key'] = df['actual_file_name'].apply(build_standardized_file_key)
    return df.reset_index(drop=True)


# Load all sheets
project_df, questionnaire_df = load_questionnaire_sheet(FILEPATH)
file_information_df           = load_file_information_sheet(FILEPATH)
metadata_mapping_df           = load_metadata_mapping_sheet(FILEPATH, file_information_df)

print(f'Sheet1 — project: {project_df.shape}, questionnaire: {questionnaire_df.shape}')
print(f'Sheet2 — file info: {file_information_df.shape}')
print(f'Sheet3 — mapping: {metadata_mapping_df.shape}')


Sheet1 — project: (5, 3), questionnaire: (52, 3)
Sheet2 — file info: (13, 26)
Sheet3 — mapping: (64, 14)


In [8]:
def extract_project_metadata(project_df):
    metadata = {}
    for _, row in project_df.iterrows():
        key = normalize_key(str(row['field']))
        value = row['value']
        if pd.notna(value):
            metadata[key] = str(value).strip()
    return metadata

def extract_governance_attributes(questionnaire_df):
    gov = {}
    for _, row in questionnaire_df.iterrows():
        cat  = str(row['category']).strip().lower()
        desc = str(row['description']).strip().lower()
        resp = row['response']
        if pd.isna(resp): continue
        resp = str(resp).strip()
        if cat == 'source name':                                         gov['source_name']             = resp
        if cat == 'business line':                                       gov['business_line']           = resp
        if cat == 'data access' and 'host type' in desc:                gov['host_type']               = resp
        if cat == 'data access' and 'delivery method' in desc:          gov['delivery_method']         = resp
        if cat == 'availability' and 'how often' in desc:               gov['data_frequency']          = resp
        if cat == 'file db information' and 'security' in desc.lower(): gov['security_classification'] = resp
        if cat == 'environments':                                        gov['environments']            = resp
        if cat == 'data retention period':                               gov['data_retention_period']   = resp
        if cat == 'source owner approval':                               gov['source_owner_approval']   = resp
    return gov

project_metadata          = extract_project_metadata(project_df)
governance_attributes     = extract_governance_attributes(questionnaire_df)
global_governance_context = {**project_metadata, **governance_attributes}

print('Global Governance Context:')
for k, v in global_governance_context.items():
    print(f'  {k}: {v}')


Global Governance Context:
  project_name: MBP 3.11 Upgrade
  planview_project_id: 8679
  impact_score: 90
  strategic_initiative_program_name: EDP
  enactment_pod_name: Ancillary Integration
  source_name: FIS Profile
  business_line: Consumer
  host_type: External
  delivery_method: GIS
  data_frequency: Daily
  environments: PROD,QA
  data_retention_period: Default 7 year
  source_owner_approval: Pending approval from Source Owner


In [9]:
def build_file_tree(file_information_df, metadata_mapping_df):
    file_tree = {}
    for _, row in file_information_df.iterrows():
        key = row['standardized_file_key']
        file_tree[key] = {'file_info': row.to_dict(), 'mappings': {}}
    for _, row in metadata_mapping_df.iterrows():
        key         = row['standardized_file_key']
        record_type = str(row.get('record_type', 'UNKNOWN'))
        table_name  = row.get('table_name', 'UNKNOWN')
        if key not in file_tree: continue
        if record_type not in file_tree[key]['mappings']:
            file_tree[key]['mappings'][record_type] = {}
        if table_name not in file_tree[key]['mappings'][record_type]:
            file_tree[key]['mappings'][record_type][table_name] = []
        file_tree[key]['mappings'][record_type][table_name].append(row.to_dict())
    return file_tree

file_tree = build_file_tree(file_information_df, metadata_mapping_df)
print(f'File tree: {len(file_tree)} files')
print(f'Keys: {list(file_tree.keys())}')


File tree: 13 files
Keys: ['mbp_accountextract_4300', 'mbp_arrangement_extract_delta_4300', 'mbp_ext_att_ar_instance_extract_delta_4300', 'mbp_ext_att_ar_instance_extract_delta_totals_4300', 'mbp_ext_att_def_extract_delta_4300', 'mbp_ext_att_def_extract_delta_totals_4300', 'mbp_ext_att_ip_instance_extract_delta_4300', 'mbp_ext_att_ip_instance_extract_delta_totals_4300', 'mbp_involved_party_extract_delta_4300', 'mbp_involved_party_extract_delta_totals_4300', 'mbp_ip_ar_rltnp_extract_delta_4300', 'mbp_ip_ip_rltnp_extract_delta_4300', 'mbp_transactionextract_4300']


In [10]:
# ── GOVERNANCE CHUNKS (Sheet 1) ──
def build_governance_chunks(project_df, questionnaire_df, global_governance_context):
    chunks = []
    project_lines = [
        f"{row['field']}: {row['value']}"
        for _, row in project_df.iterrows() if pd.notna(row['value'])
    ]
    chunks.append({
        'content': '\n'.join(project_lines),
        'metadata': {
            'domain': 'governance', 'section': 'project_overview',
            'source_name': global_governance_context.get('source_name', ''),
            'project_name': global_governance_context.get('project_name', '')
        }
    })
    for category, group in questionnaire_df.groupby('category'):
        lines = [
            f"{row['description']}: {row['response']}"
            for _, row in group.iterrows() if pd.notna(row['response'])
        ]
        if not lines: continue
        content = f"Category: {category}\n" + '\n'.join(lines)
        chunks.append({
            'content': content,
            'metadata': {
                'domain': 'governance', 'section': 'questionnaire',
                'category': category.strip().lower(),
                'source_name': global_governance_context.get('source_name', ''),
                'project_name': global_governance_context.get('project_name', '')
            }
        })
    return chunks


# ── FILE INFORMATION CHUNKS (Sheet 2) ──
def build_file_information_chunks(file_information_df):
    chunks = []
    for _, row in file_information_df.iterrows():
        lines = [
            f"File Name: {row.get('file_name')}",
            f"Description: {row.get('file_description_include_subject_area')}",
            f"Transmission Type: {row.get('transmission_type_gis_ndm_other')}",
            f"Frequency: {row.get('frequency_daily_weekly_monthly')}",
            f"Load Type: {row.get('full_delta')}",
            f"Autosys Start: {row.get('autosys_start')}",
            f"Schema: {row.get('datalake_schema_name_raw_cl')}",
            f"Encrypted: {row.get('encrypted_y_n')}",
            f"Compressed: {row.get('compressed_y_n')}",
            f"Producer Mailbox: {row.get('original_producer_mail_box')}",
            f"Consumer Mailbox: {row.get('consumer_mail_box')}",
            f"File Size: {row.get('file_size_bytes_')}",
            f"Holiday Rules: {row.get('holiday_rules')}",
            f"In Scope Acquisition: {row.get('in_scope_for_acquisition_y_n_if_n_then_reason')}",
            f"In Scope Ingestion: {row.get('in_scope_for_ingestion_y_n_if_n_then_reason')}",
        ]
        content = '\n'.join([l for l in lines if 'None' not in str(l)])
        metadata = {
            'domain':       'file_information',
            'file_key':     str(row.get('standardized_file_key', '')),
            'frequency':    str(row.get('frequency_daily_weekly_monthly', '')),
            'load_type':    str(row.get('full_delta', '')),
            'transmission': str(row.get('transmission_type_gis_ndm_other', '')),
            'schema':       str(row.get('datalake_schema_name_raw_cl', ''))
        }
        chunks.append({'content': content, 'metadata': metadata})
    return chunks


# ── TECHNICAL MAPPING CHUNKS (Sheet 3) ──
def build_technical_chunks(file_tree):
    chunks = []
    for file_key, file_data in file_tree.items():
        file_info = file_data['file_info']
        mappings  = file_data['mappings']
        for record_type, tables in mappings.items():
            for table_name, fields in tables.items():
                lines = [
                    f"File: {file_info.get('file_name')}",
                    f"Record Type: {record_type}",
                    f"Target Schema: {fields[0].get('schema_name') if fields else ''}",
                    f"Target Table: {table_name}",
                    f"Frequency: {file_info.get('frequency_daily_weekly_monthly')}",
                    f"Transmission: {file_info.get('transmission_type_gis_ndm_other')}",
                    f"Load Type: {file_info.get('full_delta')}",
                    '\nFields:'
                ]
                for f in fields:
                    lines.append(
                        f"  - {f.get('field_name')} ({f.get('field_type')}, "
                        f"max={f.get('max_length')}): "
                        f"{f.get('_field_description', f.get('field_description', ''))}"
                    )
                content  = '\n'.join(lines)
                metadata = {
                    'domain':       'technical_mapping',
                    'file_key':     file_key,
                    'record_type':  str(record_type),
                    'schema':       str(fields[0].get('schema_name', '')) if fields else '',
                    'table':        str(table_name),
                    'frequency':    str(file_info.get('frequency_daily_weekly_monthly', '')),
                    'load_type':    str(file_info.get('full_delta', '')),
                    'transmission': str(file_info.get('transmission_type_gis_ndm_other', ''))
                }
                chunks.append({'content': content, 'metadata': metadata})
    return chunks


governance_chunks = build_governance_chunks(project_df, questionnaire_df, global_governance_context)
file_chunks       = build_file_information_chunks(file_information_df)
technical_chunks  = build_technical_chunks(file_tree)
all_chunks        = governance_chunks + file_chunks + technical_chunks

print(f'Governance: {len(governance_chunks)}, File: {len(file_chunks)}, Technical: {len(technical_chunks)}')
print(f'Total chunks: {len(all_chunks)}')


Governance: 26, File: 13, Technical: 13
Total chunks: 52


In [11]:
def validate_chunks(chunks):
    for i, chunk in enumerate(chunks):
        assert chunk.get('content'),            f'Chunk {i} has empty content'
        assert chunk.get('metadata'),           f'Chunk {i} missing metadata'
        assert 'domain' in chunk['metadata'],   f'Chunk {i} missing domain key'
    domain_dist = Counter(c['metadata']['domain'] for c in chunks)
    print(f'All {len(chunks)} chunks valid.')
    print(f'Domain distribution: {dict(domain_dist)}')

validate_chunks(all_chunks)


All 52 chunks valid.
Domain distribution: {'governance': 26, 'file_information': 13, 'technical_mapping': 13}


In [12]:
embedding_model = SentenceTransformer(EMBED_MODEL)
print(f'Embedding model loaded: {EMBED_MODEL}')

# Works for both old and new chromadb versions
try:
    chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
    print('Using PersistentClient (chromadb >= 0.4.x)')
except AttributeError:
    from chromadb.config import Settings
    chroma_client = chromadb.Client(Settings(
        persist_directory=CHROMA_DIR,
        is_persistent=True
    ))
    print('Using legacy Client (chromadb < 0.4.x)')

try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print('Deleted existing collection — fresh start.')
except:
    pass

collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
print(f'Collection ready: {COLLECTION_NAME}')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\devan\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\devan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: all-MiniLM-L6-v2
Using PersistentClient (chromadb >= 0.4.x)
Collection ready: mbp_rag_collection_v2


In [13]:
documents  = [c['content']  for c in all_chunks]
metadatas  = [c['metadata'] for c in all_chunks]
ids        = [str(uuid.uuid4()) for _ in all_chunks]

print('Generating embeddings — please wait...')
embeddings = embedding_model.encode(
    documents, convert_to_numpy=True, show_progress_bar=True
).tolist()

collection.add(
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings,
    ids=ids
)
print(f'Inserted {len(documents)} chunks into ChromaDB.')


Generating embeddings — please wait...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Inserted 52 chunks into ChromaDB.


In [14]:
available_file_keys = list(file_tree.keys())

GOVERNANCE_KEYWORDS = [
    'vendor', 'risk', 'contact', 'retention', 'business line', 'security',
    'approval', 'availability', 'environment', 'sourcing', 'strategy',
    'masking', 'dr', 'disaster recovery', 'financial', 'audience',
    'party data', 'documentation', 'change management', 'quality issue',
    'products', 'subject area', 'ad group'
]
TECHNICAL_KEYWORDS = [
    'field', 'column', 'record type', 'datatype', 'table', 'schema',
    'mapping', 'target table', 'max length', 'column name'
]
FILE_KEYWORDS = [
    'frequency', 'transmission', 'encrypted', 'compressed',
    'autosys', 'load type', 'mailbox', 'file size', 'holiday',
    'delta', 'full', 'producer', 'consumer', 'in scope'
]

def detect_domain(query: str) -> tuple:
    q = query.lower()
    gov_score  = sum(1 for w in GOVERNANCE_KEYWORDS  if w in q)
    tech_score = sum(1 for w in TECHNICAL_KEYWORDS   if w in q)
    file_score = sum(1 for w in FILE_KEYWORDS         if w in q)
    scores = {'governance': gov_score, 'technical_mapping': tech_score, 'file_information': file_score}
    best_domain = max(scores, key=scores.get)
    best_score  = scores[best_domain]
    if best_score == 0:
        return 'governance', 0.0
    confidence = best_score / (sum(scores.values()) + 1e-9)
    return best_domain, round(confidence, 2)

def detect_file_key(query: str) -> str:
    query_clean = re.sub(r'[^a-z0-9]', '', query.lower())
    best_match, best_score = None, 0
    for key in available_file_keys:
        key_parts = key.split('_')
        score = sum(2 for part in key_parts if len(part) > 3 and part in query_clean)
        if score > best_score:
            best_score = score
            best_match = key
    return best_match if best_score >= 2 else None

def detect_record_type(query: str) -> str:
    match = re.search(r'\b(\d{4,5})\b', query)
    return match.group() if match else None

def build_chroma_filter(domain=None, file_key=None, record_type=None):
    conditions = []
    if domain:      conditions.append({'domain':      {'$eq': domain}})
    if file_key:    conditions.append({'file_key':    {'$eq': file_key}})
    if record_type: conditions.append({'record_type': {'$eq': record_type}})
    if not conditions: return None
    if len(conditions) == 1: return conditions[0]
    return {'$and': conditions}

print('Query router ready.')


Query router ready.


In [15]:
try:
    reranker     = CrossEncoder(RERANK_MODEL)
    USE_RERANKER = True
    print(f'Reranker loaded: {RERANK_MODEL}')
except Exception as e:
    reranker     = None
    USE_RERANKER = False
    print(f'Reranker unavailable ({e}), using semantic-only top-k.')


def retrieve_chunks(query: str, metadata_filter=None, top_k=TOP_K_RETRIEVE):
    query_embedding = embedding_model.encode(query, convert_to_numpy=True).tolist()
    kwargs = {'query_embeddings': [query_embedding], 'n_results': top_k}
    if metadata_filter:
        kwargs['where'] = metadata_filter
    results = collection.query(**kwargs)
    return results['documents'][0], results['metadatas'][0]


def rerank_chunks(query, docs, metas, top_k=TOP_K_RERANK):
    if not USE_RERANKER or len(docs) <= top_k:
        return docs[:top_k], metas[:top_k]
    scores = reranker.predict([(query, doc) for doc in docs])
    ranked = sorted(zip(scores, docs, metas), reverse=True)
    return [d for _, d, _ in ranked[:top_k]], [m for _, _, m in ranked[:top_k]]


def route_and_retrieve(query: str):
    domain, confidence = detect_domain(query)
    file_key           = detect_file_key(query)
    record_type        = detect_record_type(query)
    metadata_filter    = build_chroma_filter(domain=domain, file_key=file_key, record_type=record_type)
    docs, metas        = retrieve_chunks(query, metadata_filter=metadata_filter, top_k=TOP_K_RETRIEVE)
    # Fallback if too few results
    if len(docs) < 2:
        fallback_filter = build_chroma_filter(domain=domain)
        docs, metas     = retrieve_chunks(query, metadata_filter=fallback_filter, top_k=TOP_K_RETRIEVE)
    docs, metas = rerank_chunks(query, docs, metas, top_k=TOP_K_RERANK)
    return docs, metas, domain, confidence, file_key, record_type

print('Retrieval + reranking pipeline ready.')


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

C:\Users\devan\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\devan\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
Retrieval + reranking pipeline ready.


In [16]:
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    region_name=BEDROCK_REGION
)

SYSTEM_PROMPT = """You are a Data Lake Governance and Technical Metadata Specialist for the MBP 3.11 Upgrade project (FIS Profile source).

STRICT RULES:
1. Answer ONLY using the provided CONTEXT. Never use outside knowledge.
2. Do NOT infer or fabricate any missing details.
3. If the answer is not in context, respond exactly:
   "The requested information is not available in the provided Data Lake documentation."
4. Be professional and structured. Use bullet points when listing multiple items.
5. For file questions: always include file name, frequency, and schema.
6. For field questions: always include field name, data type, and max length.
"""

def generate_answer(query: str, docs: list, conversation_history: list = None) -> str:
    context  = '\n---\n'.join(docs)
    messages = []
    if conversation_history:
        for turn in conversation_history[-4:]:
            messages.append({'role': turn['role'], 'content': turn['content']})
    full_prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"USER QUESTION:\n{query}\n\nAnswer:"
    )
    messages.append({'role': 'user', 'content': full_prompt})
    body = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': 800,
        'temperature': 0.0,
        'messages': messages
    }
    resp = bedrock_runtime.invoke_model(
        modelId=BEDROCK_MODEL_ID,
        body=json.dumps(body)
    )
    return json.loads(resp['body'].read())['content'][0]['text']

print('Bedrock client ready.')


Bedrock client ready.


In [17]:
def log_conversation(query, answer, domain, confidence, file_key, record_type):
    file_exists = os.path.exists(LOG_FILE)
    with open(LOG_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'timestamp', 'query', 'answer', 'domain', 'confidence', 'file_key', 'record_type'
        ])
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'timestamp':   datetime.now().isoformat(),
            'query':       query,
            'answer':      answer,
            'domain':      domain,
            'confidence':  confidence,
            'file_key':    file_key or '',
            'record_type': record_type or ''
        })

print(f'Logger ready -> {LOG_FILE}')


Logger ready -> chat_log.csv


In [18]:
conversation_history = []

def ask_rag_system(query: str, verbose: bool = True) -> str:
    """
    Full advanced RAG pipeline:
      1. Route -> detect domain, file_key, record_type
      2. Retrieve top-K chunks (semantic + metadata filter)
      3. Rerank with CrossEncoder
      4. Generate answer via Bedrock Claude 3 Haiku
      5. Store in conversation memory + CSV log
    """
    docs, metas, domain, confidence, file_key, record_type = route_and_retrieve(query)

    if verbose:
        print(f'[Router]    domain={domain} | confidence={confidence} | '
              f'file_key={file_key} | record_type={record_type}')
        print(f'[Retrieval] {len(docs)} chunks after reranking')

    answer = generate_answer(query, docs, conversation_history)

    conversation_history.append({'role': 'user',      'content': query})
    conversation_history.append({'role': 'assistant', 'content': answer})
    if len(conversation_history) > 20:
        conversation_history.pop(0)
        conversation_history.pop(0)

    log_conversation(query, answer, domain, confidence, file_key, record_type)
    return answer


def reset_conversation():
    global conversation_history
    conversation_history = []
    print('Conversation history cleared.')

print('RAG pipeline ready. Use ask_rag_system("your question") to query.')


RAG pipeline ready. Use ask_rag_system("your question") to query.


In [19]:
test_questions = [
    'What is the frequency of MBP Account Extract?',
    'Who is the vendor for this source?',
    'What fields are in record type 9000 of MBP Account Extract?',
    'What is the data retention period?',
    'Is MBP Account Extract encrypted?',
    'What are the environments available?',
    'What is the security classification of this source?',
    'What is the autosys start time for the arrangement file?',
    'What is the target table for MBP Transaction Extract?',
    'What is the CEO salary?',  # Out-of-scope test
]

reset_conversation()
for q in test_questions:
    print(f'\n{"="*70}')
    print(f'Q: {q}')
    answer = ask_rag_system(q)
    print(f'A: {answer}')


Conversation history cleared.

Q: What is the frequency of MBP Account Extract?
[Router]    domain=file_information | confidence=1.0 | file_key=mbp_accountextract_4300 | record_type=None
[Retrieval] 3 chunks after reranking


NoCredentialsError: Unable to locate credentials

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

SAMPLE_QUESTIONS = [
    'What is the frequency of MBP Account Extract?',
    'What is the transmission type of MBP Transaction Extract?',
    'What fields are in record type 9000 of MBP Account Extract?',
    'Who is the vendor?',
    'What is the data retention period?',
    'What environments exist for this source?',
    'What is the autosys start time of the arrangement file?',
    'What is the sourcing strategy?',
    'What is the security classification?',
    'What are the change management contacts?',
]

sample_area  = widgets.Textarea(
    value='\n'.join(SAMPLE_QUESTIONS),
    description='Samples:',
    layout=widgets.Layout(width='100%', height='180px')
)
question_box = widgets.Textarea(
    placeholder='Type your question here and click Ask...',
    description='Question:',
    layout=widgets.Layout(width='100%', height='80px')
)
ask_btn  = widgets.Button(description='Ask',           button_style='primary')
clear_btn= widgets.Button(description='Clear History', button_style='warning')
out_box  = widgets.Output()

def on_ask(b):
    with out_box:
        clear_output()
        q = question_box.value.strip()
        if not q:
            print('Please enter a question.')
            return
        print(f'Processing: {q}...')
        print(f'\nAnswer:\n{ask_rag_system(q)}')

def on_clear(b):
    with out_box:
        clear_output()
        reset_conversation()

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

display(
    widgets.HTML('<h3>MBP 3.11 Upgrade — Data Lake RAG Chatbot</h3>'),
    sample_area,
    question_box,
    widgets.HBox([ask_btn, clear_btn]),
    out_box
)


In [ ]:
# Uncomment to launch in browser tab

# import gradio as gr
#
# def gradio_chat(user_message, history):
#     answer = ask_rag_system(user_message)
#     history.append((user_message, answer))
#     return '', history
#
# with gr.Blocks(title='MBP DataLake RAG Chatbot') as demo:
#     gr.Markdown('## MBP 3.11 Upgrade — Data Lake RAG Chatbot')
#     gr.Markdown('Ask anything about files, fields, governance, vendors, contacts, and more.')
#     chatbot   = gr.Chatbot(height=450)
#     msg_input = gr.Textbox(placeholder='Ask anything about the Data Lake...', label='Question')
#     clear_btn = gr.Button('Clear History')
#     msg_input.submit(gradio_chat, [msg_input, chatbot], [msg_input, chatbot])
#     clear_btn.click(lambda: [], None, [chatbot])
#
# demo.launch(share=False)

print('Gradio UI ready — uncomment to launch.')


In [ ]:
if os.path.exists(LOG_FILE):
    log_df = pd.read_csv(LOG_FILE)
    print(f'Total logged queries: {len(log_df)}')
    display(log_df[['timestamp', 'query', 'domain', 'confidence', 'file_key']].tail(10))
else:
    print('No log file yet. Run some queries first.')
